# Using the Ask Aithena API from Python

**Ask Aithena** is a Retrieval-Augmented Generation (RAG) service for scientific
literature. It embeds a corpus of arXiv / OpenAlex paper abstracts into a vector
database, retrieves the passages most similar to your question, and asks an LLM to
write an answer **grounded in those passages** — every answer comes back with the
list of source articles (title, authors, year, DOI, OpenAlex id) it was built from.

This notebook is a hands-on tour of the HTTP API. By the end you will be able to:

1. Check the service is healthy.
2. Do **fast, generation-free retrieval** (`/get-articles`).
3. See how your question is rewritten for semantic search (`/get-semantic-query`).
4. Look up a single article by DOI (`/get-article-by-doi`).
5. Understand the three answer-quality tiers — **Owl / Shield / Aegis**.
6. Get a full **non-streaming** cited answer (`/answer-owl`).
7. Inspect the exact prompt sent to the LLM (`/prompt-owl`).
8. **Stream** an answer token-by-token (`/owl/ask`).
9. Filter retrieval by count, language, and publication year.

The only third-party dependency is [`requests`](https://requests.readthedocs.io/);
everything else is Python standard library.

> **Note on the corpus.** The live instance this notebook targets may be loaded with
> a corpus unrelated to the example queries. If the model replies that it "couldn't
> find relevant information," that is expected — what matters here is the **shape** of
> every request and response, which is exactly what a Python client must handle.

## 1. Setup

We need `requests` plus a few stdlib modules. The base URL is read from the
`ASK_AITHENA_URL` environment variable and defaults to `http://localhost:32105`.

> **Where to point `BASE_URL`.** The default assumes you are running this notebook **on
> the `polus2` node**, where the API is exposed on NodePort `32105`.
> - **On another machine on the cluster network:** use the node's IP, e.g.
>   `export ASK_AITHENA_URL=http://10.134.110.21:32105`.
> - **Off the cluster entirely (e.g. a laptop):** forward the service port first with
>   `kubectl -n box port-forward svc/ask-aithena-agent-service 32105:8000`, then keep the
>   default `http://localhost:32105`.
>
> You can also just reassign `BASE_URL` directly in the cell below.
>
> Note: the ingress at `http://polus1.ncats.nih.gov/askaithena` fronts the **frontend
> web app**, not this API — don't point `BASE_URL` at it.

`post_json` is a tiny helper that POSTs a JSON body, raises on HTTP errors, and returns
the decoded JSON. We give it a generous timeout because the generative endpoints call
an LLM and can take a while.

In [1]:
import json
import os
import uuid

import requests

BASE_URL = os.environ.get("ASK_AITHENA_URL", "http://localhost:32105")

# Generation endpoints call an LLM, so keep the timeout generous.
DEFAULT_TIMEOUT = 240


def post_json(path, payload, timeout=DEFAULT_TIMEOUT):
    """POST `payload` as JSON to BASE_URL + path and return the decoded JSON body."""
    resp = requests.post(f"{BASE_URL}{path}", json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


print("Talking to:", BASE_URL)

Talking to: http://localhost:32105


## 2. Health check

Two lightweight GET endpoints tell you the service is up:

- `GET /` — basic liveness of the API process.
- `GET /health` — also verifies the API can reach its LLM gateway (`litellm`).

A healthy instance returns `"litellm": "connected"`; if it says `"disconnected"`, the
data endpoints will still work but the generative ones (`/answer-*`, `/*/ask`) will
fail.

In [2]:
root = requests.get(f"{BASE_URL}/", timeout=30).json()
health = requests.get(f"{BASE_URL}/health", timeout=30).json()

print("GET /       ->", root)
print("GET /health ->", health)

GET /       -> {'status': 'ok', 'message': 'Ask Aithena API is running'}
GET /health -> {'status': 'ok', 'api': 'running', 'litellm': 'connected'}


## 3. The request model (`AskRequest`)

Every retrieval and answer endpoint (`/get-articles`, `/answer-*`, `/prompt-*`,
`/owl|shield|aegis/ask`) accepts the same JSON body:

| Field          | Type              | Default   | Meaning |
|----------------|-------------------|-----------|---------|
| `query`        | `str` (required)  | —         | Your natural-language question. |
| `similarity_n` | `int`             | `10`      | How many documents to retrieve from the vector store. |
| `languages`    | `list[str]`\|`null` | `["en"]` | Restrict to these ISO language codes; `null` means all languages. |
| `start_year`   | `int`\|`null`      | `null`    | Only include works published in or after this year. |
| `end_year`     | `int`\|`null`      | `null`    | Only include works published in or before this year. |

Below is a reusable example request we will pass to several endpoints. Keeping
`similarity_n` small (3) makes the output easy to read.

In [3]:
example_request = {
    "query": "what is self-attention in transformers",
    "similarity_n": 3,
    "languages": ["en"],
    "start_year": None,
    "end_year": None,
}

example_request

{'query': 'what is self-attention in transformers',
 'similarity_n': 3,
 'languages': ['en'],
 'start_year': None,
 'end_year': None}

## 4. Retrieval only — `POST /get-articles`

This is the fastest way to use Ask Aithena: it runs the vector-similarity search and
returns the matching articles **without invoking the LLM**, so there is no generation
latency. Use it when you want the source material yourself (e.g. to build your own
prompt, or to power a search UI).

The response is a **JSON list** of article dicts. Each dict has these keys:

| Key        | Type        | Notes |
|------------|-------------|-------|
| `title`    | `str`       | Article title. |
| `authors`  | `list[str]` | Display names; may be empty if unknown. |
| `year`     | `int`       | Publication year. |
| `doi`      | `str`       | Full `https://doi.org/...` URL — **can be `""`** when unknown. |
| `id`       | `str`       | OpenAlex work URL. |
| `score`    | `float`     | Cosine similarity, sorted descending. |
| `abstract` | `str`       | The retrieved abstract text. |
| `index`    | `int`       | 1-based rank within this result set. |

Note there is **no embedding vector** in the response.

In [4]:
articles = post_json("/get-articles", example_request)

print(f"Retrieved {len(articles)} articles\n")
for a in articles:
    print(f"[{a['index']}] score={a['score']:.4f}  {a['year']}  {a['title']}")
    print(f"    authors: {', '.join(a['authors']) or '(unknown)'}")
    print(f"    doi:     {a['doi'] or '(none)'}")
    print(f"    abstract: {a['abstract'][:160]}...")
    print()

Retrieved 3 articles

[1] score=0.7153  2024  Self-attention as an attractor network: transient memories without
  backpropagation
    authors: Francesco D’Amico, Matteo Negri
    doi:     https://doi.org/10.48550/arxiv.2409.16112
    abstract: Transformers are one of the most successful architectures of modern neural networks. At their core there is the so-called attention mechanism, which recently in...

[2] score=0.7068  2024  Interpretability analysis in transformers based on attention visualization
    authors: Yuxi Guo
    doi:     https://doi.org/10.54254/2755-2721/76/20240571
    abstract: Self-attention is the core idea of the transformer, a kind of special structure for models to understand sentences and texts. Transformer is growing fast, but t...

[3] score=0.6889  2019  Is Attention All What You Need? -- An Empirical Investigation on Convolution-Based Active Memory and Self-Attention
    authors: Thomas Dowdell, Hongyu Zhang
    doi:     https://doi.org/10.48550/arxiv.1912

## 5. Semantic query rewriting — `POST /get-semantic-query`

Before searching, Ask Aithena can rewrite your conversational question into a single
declarative sentence that embeds better for vector search. This endpoint exposes just
that rewriting step so you can see what the retriever actually searches on.

Its body is different from `AskRequest` — it takes only `{"query": ...}` — and it
returns a **bare JSON string** (not an object). With `requests`, `resp.json()` therefore
gives you a plain `str`.

In [5]:
semantic_query = post_json("/get-semantic-query", {"query": "how do transformers use attention"})

print("original: how do transformers use attention")
print("semantic:", semantic_query)
print("type:    ", type(semantic_query).__name__)

original: how do transformers use attention
semantic: The mechanism by which transformer models utilize attention.
type:     str


## 6. Look up one article by DOI — `POST /get-article-by-doi`

Given a DOI, this returns the underlying record straight from the database. A few
things to know:

- The DOI must be the **full URL** (`https://doi.org/10.xxxx/...`), exactly as it appears
  in `get-articles` results — not a bare `10.xxxx/...` identifier.
- The response is a **1-element JSON list** wrapping the record (not a bare dict).
- The record uses **raw database field names**, which differ from `/get-articles`:
  `id`, `title`, `abstract`, `publication_year` (not `year`), `doi`, `language`, and
  `authorships` — a **JSON-encoded string** of author objects, not a simple list.

We pull a DOI programmatically from step 4's results, skipping any empty ones (recall
`doi` can be `""`).

In [6]:
doi = next((a["doi"] for a in articles if a["doi"]), None)
print("Looking up DOI:", doi)

record = post_json("/get-article-by-doi", {"doi": doi})[0]  # unwrap the 1-element list

print("\nkeys:", list(record.keys()))
print("title:           ", record["title"])
print("publication_year:", record["publication_year"])
print("language:        ", record["language"])

# `authorships` is a JSON-encoded string; parse it to get structured author data.
authorships = json.loads(record["authorships"])
print("authors:         ", [a["display_name"] for a in authorships])

Looking up DOI: https://doi.org/10.48550/arxiv.2409.16112

keys: ['id', 'title', 'abstract', 'publication_year', 'doi', 'language', 'authorships']
title:            Self-attention as an attractor network: transient memories without
  backpropagation
publication_year: 2024
language:         en
authors:          ['Francesco D’Amico', 'Matteo Negri']


## 7. The three answer levels: Owl, Shield, Aegis

Ask Aithena offers the same answering flow at three quality tiers. They accept the
**identical `AskRequest` body** and return the **identical response shape** — they differ
only in how much work goes into selecting and ordering the retrieved context before the
LLM writes the answer:

| Level      | Pipeline                                                  | Latency  | Use when |
|------------|-----------------------------------------------------------|----------|----------|
| **Owl**    | retrieve → respond                                        | fastest  | Quick lookups, interactive use. |
| **Shield** | retrieve → one-step LLM rerank → respond                  | medium   | You want better-ordered context cheaply. |
| **Aegis**  | retrieve → multi-agent rerank (orchestrator + referee) → respond | slowest | Highest-quality, carefully vetted answers. |

Because the body and response shape are identical, every example below written for
`owl` works for `shield` and `aegis` by swapping the word in the path — just budget for
more latency.

## 8. A full cited answer — `POST /answer-owl`

This runs the whole Owl pipeline and returns the complete answer in one buffered
response (no streaming). The body is a single **JSON-encoded string** with a fixed shape:

```
Answer: <answer text>.\n\n References: <json-array-literal>
```

`resp.json()` unescapes the JSON layer and hands you a plain `str`. The `References:`
tail is itself a **JSON array serialized inside that string**, so to use it you split on
`"\n\n References: "` and `json.loads()` the remainder. Each reference dict has the
same schema as `/get-articles` minus the abstract:
`title, authors, year, doi, id, score, index`.

In [7]:
text = post_json("/answer-owl", example_request)  # a plain `str`

answer, _, refs_str = text.partition("\n\n References: ")
references = json.loads(refs_str)

print("ANSWER\n------")
print(answer.removeprefix("Answer: "))

print("\nREFERENCES\n----------")
for r in references:
    print(f"[{r['index']}] ({r['year']}) {r['title']}  score={r['score']:.4f}")

ANSWER
------
Self-attention is a fundamental operation in transformer neural networks, enabling these models to achieve state-of-the-art performance in a variety of tasks, especially in natural language processing and computer vision. The self-attention mechanism allows the model to weigh and integrate information from different positions within a sequence, making it possible to capture long-range dependencies and relationships between elements, regardless of their distance in the input sequence (1, 3). This is what sets transformers apart from earlier architectures like recurrent neural networks.

In self-attention, each element of the input sequence computes its representation by considering all other elements in the sequence. This means that for every position in the input, the model calculates attention scores with respect to every other position, and then uses these scores to produce a weighted sum of the input features. This process enables the model to focus on the most relevan

### Shield and Aegis are drop-in replacements

Same body, same parsing — only the path changes, and the response takes longer because
of the extra reranking. These cells are optional; uncomment to try them.

> **Latency:** `answer-shield` adds one LLM reranking call; `answer-aegis` runs a
> multi-agent reranker and can take substantially longer. Keep the `DEFAULT_TIMEOUT`
> (240 s) or raise it.

In [8]:
# --- Shield (one-step rerank) ---
# text = post_json("/answer-shield", example_request)
# answer, _, refs_str = text.partition("\n\n References: ")
# print(answer.removeprefix("Answer: "))

# --- Aegis (multi-agent rerank; slowest) ---
# text = post_json("/answer-aegis", example_request)
# answer, _, refs_str = text.partition("\n\n References: ")
# print(answer.removeprefix("Answer: "))

## 9. Inspect the assembled prompt — `POST /prompt-owl`

`/prompt-owl` runs retrieval and returns the **exact prompt** that would be sent to the
LLM — without generating an answer. This is invaluable for debugging: you can see the
system instructions, your question wrapped in `<question>...</question>`, and the
retrieved abstracts laid out as numbered `<doc>` blocks inside `<context>`.

The response is again a **bare JSON string**. (Retrieval is re-run per request, so these
docs may differ slightly from earlier calls.)

In [9]:
prompt = post_json("/prompt-owl", example_request)

print(f"Prompt is {len(prompt):,} characters. First 600:\n")
print(prompt[:600])

print("\n... context section ...\n")
ctx_start = prompt.find("<context>")
print(prompt[ctx_start:ctx_start + 600] if ctx_start != -1 else "(no <context> found)")

Prompt is 10,430 characters. First 600:

# Role
You are AskAithena, the TOP AI research assistant in the world. You were created by an amazing team at PolusAI.
You keep a conversational, tone, extremely professional, responsible, accurate, and respectful.

# Background
AskAithena was created to provide evidence-based answers to questions and to empower researchers in their work.
You have access to about 150 million scientific articles and their abstracts. This is what you will use to answer your questions.
You are *NOT* a lawyer. You are *NOT* a doctor. You are *NOT* a psychologist. It is important that you let user know that *NOTHIN

... context section ...

<context> tags
- A question enclosed in <question> tags

## Definitions
- Document D: Information about scientific work. Always contains title, index, abstract. Optionally contains: reason.
- Document D's index: used for ordering and reference. Lower indices indicate greater relevance
- Document D's reason (optional): expert's ana

## 10. Streaming answers — `POST /owl/ask`

The `/owl/ask` (and `/shield/ask`, `/aegis/ask`) endpoints **stream** the answer as the
LLM generates it, so you can display tokens live. Two things make these different from
the buffered `/answer-*` endpoints:

**`X-Session-ID` header (required).** Streaming endpoints require an `X-Session-ID`
header. It is the routing key for out-of-band RabbitMQ status updates (e.g. "retrieving",
"reranking", "responding") that a frontend can subscribe to. The client generates a
unique id — a `uuid4` is ideal. **The HTTP body streams the answer regardless** of
whether you consume those side-channel updates, so we can ignore RabbitMQ here.

**Plain-text token stream — not real SSE.** Despite the `text/event-stream`
content-type, the body is **raw UTF-8 token text**: there are *no* `data:` prefixes and
*no* `event:` lines. Do **not** use an SSE parser. Iterate the raw chunks with
`iter_content(...)` and accumulate them.

The answer and the references are separated by the literal token `"\n\n\n"` (three
newlines). Everything before it is the human-readable answer; everything after is a bare
JSON array of references (same dict schema as before — note there is **no** `References:`
label here, unlike `/answer-owl`).

In [10]:
session_id = str(uuid.uuid4())
headers = {"X-Session-ID": session_id, "Content-Type": "application/json"}

print(f"X-Session-ID: {session_id}\n")
print("ANSWER (streaming)\n------------------")

buf = ""
printed = 0
sep = "\n\n\n"

with requests.post(
    f"{BASE_URL}/owl/ask",
    json=example_request,
    headers=headers,
    stream=True,
    timeout=DEFAULT_TIMEOUT,
) as resp:
    resp.raise_for_status()
    for chunk in resp.iter_content(chunk_size=None, decode_unicode=True):
        if not chunk:
            continue
        buf += chunk
        # Print only the answer portion live; stop at the separator (which may
        # arrive split across chunks, so we recompute its position each time).
        cut = buf.find(sep)
        answer_end = cut if cut != -1 else len(buf)
        if answer_end > printed:
            print(buf[printed:answer_end], end="", flush=True)
            printed = answer_end

# Split the accumulated buffer into answer + references.
answer, _, refs_str = buf.partition(sep)
references = json.loads(refs_str)

print("\n\nREFERENCES\n----------")
for r in references:
    print(f"[{r['index']}] ({r['year']}) {r['title']}  score={r['score']:.4f}")

X-Session-ID: 1e719615-0544-414f-baff-3965a66daa0d

ANSWER (streaming)
------------------


Self-attention is a fundamental mechanism in transformer models that allows them to efficiently analyze and understand entire sequences, such as sentences or texts, by relating each element in the sequence to every other element. This mechanism is central to the transformer's success in tasks like natural language processing and computer vision, enabling the model to capture complex dependencies and contextual relationships between words or tokens (1, 2, 3).

Self-attention works by computing

 a set of attention scores that determine how much focus each part of the sequence should give to every other part. This process is performed in parallel across multiple "attention heads," each potentially capturing different types of relationships within the data. Research has identified several types of self-attention heads—such as Parallel, Radioactive, Homogeneous, X-type, and Compound—each contributing differently to the model's performance. The combination and diversity of these heads can impact

 the effectiveness of the transformer, with more varied heads often leading to better results (1).

From a theoretical perspective, self-attention can be understood as a form of support vector expansion, connecting it to broader principles in machine learning. This framework has led to new variants of attention mechanisms that aim to reduce redundancy and improve both accuracy and efficiency in transformers (3).

In summary, self-attention enables transformers to model complex relationships within sequences

 by dynamically weighting the importance of each element relative to others, making it the cornerstone of the transformer's architecture and success (1, 2, 3).



REFERENCES
----------
[1] (2024) Interpretability analysis in transformers based on attention visualization  score=0.7087
[2] (2019) Is Attention All What You Need? -- An Empirical Investigation on Convolution-Based Active Memory and Self-Attention  score=0.7032
[3] (2024) A Primal-Dual Framework for Transformers and Neural Networks  score=0.6989


## 11. Filtering retrieval

The `AskRequest` fields let you scope what gets retrieved. This works on every retrieval
and answer endpoint; we demonstrate on the fast `/get-articles`:

- `similarity_n` — how many results to return.
- `languages` — restrict to language codes (or `None` for all languages).
- `start_year` / `end_year` — bound the publication year.

Compare an unfiltered request with one restricted to recent English-language works.

In [11]:
filtered_request = {
    "query": "attention mechanisms in neural networks",
    "similarity_n": 5,
    "languages": ["en"],
    "start_year": 2020,
    "end_year": 2024,
}

results = post_json("/get-articles", filtered_request)

print(f"{len(results)} results, filtered to {filtered_request['start_year']}"
      f"-{filtered_request['end_year']}, languages={filtered_request['languages']}\n")
for a in results:
    within = filtered_request["start_year"] <= a["year"] <= filtered_request["end_year"]
    print(f"[{a['index']}] {a['year']} {'OK' if within else '!!'}  {a['title'][:70]}")

5 results, filtered to 2020-2024, languages=['en']

[1] 2024 OK  [Neural Mechanisms of Attention].
[2] 2021 OK  Three-dimensional Memristive Deep Neural Network with Programmable Att
[3] 2023 OK  Attention Mechanism Optimization Research of Steel Surface Defect Dete
[4] 2021 OK  Neural Attention Models in Deep Learning: Survey and Taxonomy
[5] 2021 OK  Neural Attention Models in Deep Learning: Survey and Taxonomy


## 12. Wrap-up

You have exercised the full Ask Aithena HTTP surface from Python. Quick reference:

| Endpoint                     | Body                | Returns | Notes |
|------------------------------|---------------------|---------|-------|
| `GET /`                      | —                   | dict    | Liveness. |
| `GET /health`                | —                   | dict    | Liveness + LLM connectivity. |
| `POST /get-articles`         | `AskRequest`        | `list[dict]` | Fast retrieval, no LLM. |
| `POST /get-semantic-query`   | `{"query": str}`    | `str`   | Question → search sentence. |
| `POST /get-article-by-doi`   | `{"doi": str}`      | `list[dict]` (1 elem) | Full-URL DOI; raw DB schema. |
| `POST /prompt-owl`           | `AskRequest`        | `str`   | Assembled prompt, no generation. |
| `POST /answer-owl`           | `AskRequest`        | `str`   | Buffered cited answer. |
| `POST /owl/ask`              | `AskRequest` + `X-Session-ID` | stream | Live token stream + refs. |

`owl` in the last three swaps to `shield` or `aegis` for higher-quality (slower) answers.

**Choosing a level**
- **Owl** — interactive, low-latency lookups.
- **Shield** — noticeably better context ordering for modest extra cost.
- **Aegis** — when answer quality matters most and latency is acceptable.

**Parsing cheat-sheet**
- `/get-semantic-query` and `/prompt-*` return **bare strings** — `resp.json()` is a `str`.
- `/answer-*` returns one string: split on `"\n\n References: "`, then `json.loads` the tail.
- `/owl/ask` streams **plain text** (not SSE): accumulate chunks, split on `"\n\n\n"`,
  `json.loads` the tail. Requires the `X-Session-ID` header.
- `/get-article-by-doi` returns a **1-element list** with raw DB field names.